In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import json
import torch
from sentence_transformers import SentenceTransformer
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import joblib
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from xgboost import XGBClassifier
import mlflow
import mlflow.xgboost
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
from mlflow import MlflowClient


#### Load Data

In [5]:
# train data
train_data=np.load("../outputs/embedded_data/train.npz")
X_train=train_data["X"]
y_train=train_data["y"]
# valid data
valid_data=np.load("../outputs/embedded_data/valid.npz")
X_valid=valid_data["X"]
y_valid=valid_data["y"]
# test data
test_data=np.load("../outputs/embedded_data/test.npz")
X_test=test_data["X"]
y_test=test_data["y"]

In [6]:
X_train_cv=np.vstack([
    X_train,
    X_valid
])

y_train_cv=np.concatenate([
    y_train,
    y_valid
])

#### Building Xgb model

In [ ]:
xgb=XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    device="cuda"
)

param_grid={
    "n_estimators": [200,300,400],
    "max_depth": [3,5,7,8],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.5,0.8,1],
    "colsample_bytree": [0.8, 1]
}

cv=StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
grid=GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=cv,
    scoring="f1_macro",
    n_jobs=1,
    verbose=3
)

grid.fit(X_train_cv, y_train_cv)

Fitting 5 folds for each of 144 candidates, totalling 720 fits


c:\Users\Kun Bi\Desktop\financial_sentiments\.senti_venv\Lib\site-packages\xgboost\core.py:729: UserWarning: [18:29:53] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[CV 1/5] END colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_estimators=200, subsample=0.5;, score=0.608 total time=   3.2s
[CV 2/5] END colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_estimators=200, subsample=0.5;, score=0.648 total time=   1.4s


#### Traking with mlflow

In [ ]:
best_params=grid.best_params_
mlflow.set_experiment("financial_sentiments")
with mlflow.start_run(run_name="xgboost_embeddings") as run:
    best_xgb=XGBClassifier(
        **best_params,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42
    )

    best_xgb.fit(X_train_cv, y_train_cv)
    mlflow.log_params(best_params)
    mlflow.log_metric("best_cv_f1_macro", grid.best_score_)
    

    y_pred=best_xgb.predict(X_test)
    y_train_pred=best_xgb.predict(X_train_cv)


    # train metrics

    train_accuracy=accuracy_score(y_train_cv,y_train_pred)
    train_f1_macro=f1_score(y_train_cv,y_train_pred, average="macro", zero_division=0)
    train_precision_macro=precision_score(y_train_cv,y_train_pred, average="macro", zero_division=0)
    train_recall_macro=recall_score(y_train_cv,y_train_pred, average="macro", zero_division=0)

    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("train_f1_macro", train_f1_macro)
    mlflow.log_metric("train_precision_macro", train_precision_macro)
    mlflow.log_metric("train_recall_macro", train_recall_macro)
    # test metrics
    test_accuracy=accuracy_score(y_test, y_pred)
    test_f1_macro=f1_score(y_test, y_pred, average="macro", zero_division=0)
    test_precision_macro=precision_score(y_test, y_pred, average="macro", zero_division=0)
    test_recall_macro=recall_score(y_test, y_pred, average="macro", zero_division=0)

    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.log_metric("test_f1_macro", test_f1_macro)
    mlflow.log_metric("test_precision_macro", test_precision_macro)
    mlflow.log_metric("test_recall_macro", test_recall_macro)
    
    # log model
    model_info=mlflow.xgboost.log_model(
        best_xgb,
        name="xgb_sentiment_model"
    )

    run_id=run.info.run_id
    
    

In [ ]:
df_metrics=pd.DataFrame({
    "train":[train_accuracy, train_f1_macro, train_precision_macro, train_recall_macro],
    "test":[test_accuracy, test_f1_macro, test_precision_macro, test_recall_macro]
}, index=["accuracy", "f1", "precision", "recall"])
df_metrics.to_csv("../outputs/metrics/xgb_metrics.csv")
df_metrics

#### Register model with MlflowClient()

In [ ]:
registered_model_xgb=mlflow.register_model(
    model_uri=model_info.model_uri,
    name="xgboost_financial_sentiment"
)
# pre-production model
client=MlflowClient()
client.set_registered_model_alias(
    name="xgboost_financial_sentiment",
    alias="candidate",
    version=registered_model_xgb.version
)

# production model
client.set_registered_model_alias(
    name="xgboost_financial_sentiment",
    alias="champion",
    version=registered_model_xgb.version
)

#### Load model from Model Registry alias

In [ ]:
loaded_xgb_model=mlflow.xgboost.load_model(
    "models:/xgboost_financial_sentiment@champion"
)

#### Loading models from mlflow

In [ ]:
model_uri=f"runs:/{run_id}/xgb_sentiment_model"
model=mlflow.xgboost.load_model(model_uri)